# Do we really need Q?

In [1]:
import torch
from transformers import AutoModelForCausalLM, GPT2Tokenizer

/Users/shre/courses/llm_deep_dive/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# load SMALL pretrained GPT-2 model and tokenizer
gpt2_orig = AutoModelForCausalLM.from_pretrained('gpt2') # this one won't be modified
gpt2      = AutoModelForCausalLM.from_pretrained('gpt2')
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

In [3]:
# use GPU
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(device)

cpu


In [4]:
# push the model to the GPU
gpt2.to(device)

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [5]:
# generate some input
x = torch.tensor(tokenizer.encode('Hello, how are you today?')).unsqueeze(0)

In [6]:
### on the CPU
torch.manual_seed(42) # seed the rng
out = gpt2_orig.generate(x, temperature=1, do_sample=True, max_length=100, 
                         pad_token_id=tokenizer.eos_token_id)
print(tokenizer.decode(out[0].tolist()))

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Hello, how are you today? I'm sorry to have asked you this."

"It's fine then," said the man. "How do you feel about that?"

The man sighed and held out his hand. "It's what I want. What's going on?"

"It's a mistake," said Mrs. Murnane. "As you would say, you have an obligation to make your needs known. If we're going to talk about your new boyfriend


In [7]:
### on the GPU
torch.manual_seed(42) # seed the rng
out = gpt2.generate(x.to(device),temperature=1,do_sample=True,max_length=100,
                    pad_token_id=tokenizer.eos_token_id)
print(tokenizer.decode(out[0].tolist()))

# Note: some differences between CPU and GPU are expected due to differential rounding/precision errors
#       but the rng seed is different on the GPU and CPU

Hello, how are you today? I'm sorry to have asked you this."

"It's fine then," said the man. "How do you feel about that?"

The man sighed and held out his hand. "It's what I want. What's going on?"

"It's a mistake," said Mrs. Murnane. "As you would say, you have an obligation to make your needs known. If we're going to talk about your new boyfriend


In [8]:
# FYI, use this code to copy the original parameters onto a modified model
gpt2.load_state_dict( gpt2_orig.state_dict() )

<All keys matched successfully>

## Replace one Q matrix

In [9]:
# size of Q matrix
n_emb = gpt2.transformer.wte.weight.shape[1]
sizeOfQ = gpt2.transformer.h[0].attn.c_attn.weight[:,:n_emb].shape
sizeOfQ

torch.Size([768, 768])

In [10]:
# get the matrix
q = gpt2.transformer.h[0].attn.c_attn.weight[:,:n_emb]

# create noise with matching statistics
mean = q.mean()
std  = q.std()
noise = torch.randn_like(q)*std + mean

# replace the values in q
gpt2.transformer.h[0].attn.c_attn.weight.data[:,:n_emb] = noise
# q.data.copy_(noise); # this also works...

In [11]:
# confirm that the weights really have changed...
print(gpt2.transformer.h[0].attn.c_attn.weight[:10,1])
print(gpt2_orig.transformer.h[0].attn.c_attn.weight[:10,1])

tensor([-0.3555, -0.4315,  0.0320, -0.0456, -0.3903,  0.4028, -0.2019,  0.0249,
         0.0897, -0.3070], grad_fn=<SelectBackward0>)
tensor([-0.2614,  0.1473,  0.0695, -0.1884,  0.1678, -0.0265, -0.0360,  0.2333,
         0.0740,  0.2161], grad_fn=<SelectBackward0>)


In [12]:
### re-run on the GPU
torch.manual_seed(42) # seed the rng
out = gpt2.generate(x.to(device), temperature=1, do_sample=True, max_length=100,
                    pad_token_id=tokenizer.eos_token_id)
print(tokenizer.decode(out[0].tolist()))

Hello, how are you today? I'm sorry to have asked you this. If you are to take the exam again, we can have an exchange of money. I wanted to go to the school and I wanted to play and I want to be part of the group. It's just being all good. I want to go to the school because the money is really good. It means I want to give back. I want to go to the school because I wanted to play. I could not


## Successively lobotomize the model :(

In [13]:
# reset the model to its original parameters
gpt2.load_state_dict( gpt2_orig.state_dict() )

<All keys matched successfully>

In [14]:
# model output pre-changes
print('** Using all original Q values:')
x = torch.tensor(tokenizer.encode('I went to the market to buy')).unsqueeze(0)
out = gpt2.generate(x.to(device),temperature=1,do_sample=True,max_length=50,
                    pad_token_id=tokenizer.eos_token_id)
print(tokenizer.decode(out[0].tolist()))


# loop over all transformer heads, replace Q, and give the same input
for hidx in range(len(gpt2.transformer.h)):

  print(f'\n\n** Now replacing Q{hidx} with noise:')

  # get the matrix
  q = gpt2.transformer.h[hidx].attn.c_attn.weight[:,:n_emb]

  # create noise with matching statistics
  mean = q.mean()
  std  = q.std()
  noise = torch.randn_like(q)*std + mean

  # replace the values in q
  gpt2.transformer.h[hidx].attn.c_attn.weight.data[:,:n_emb] = noise

  # generate the output
  out = gpt2.generate(x.to(device),temperature=1,do_sample=True,max_length=50,
                      pad_token_id=tokenizer.eos_token_id)
  print(tokenizer.decode(out[0].tolist()))

** Using all original Q values:
I went to the market to buy a large box of wine from a local company on Facebook. I had a friend get hold of it, and he was like, look, I need a ton of wine."

The company told her there were


** Now replacing Q0 with noise:
I went to the market to buy a new home.

I'm pretty sure the people in the car I'm wearing these are all going to be my mom's home.

This is what it looks like.

The best I


** Now replacing Q1 with noise:
I went to the market to buy a car for 10.


The last one of that day that I did not own that car that day and then was a very bad car. I went to the market to buy a car for car for car


** Now replacing Q2 with noise:
I went to the market to buy my favorite beer, a very strong, so. I got the best. I could I do well do, and and that had a a good on that's very strong. Then I felt like I, and that


** Now replacing Q3 with noise:
I went to the market to buy and make the most of the most of the most of the most of the most of the 